# EDA — `panel_final.parquet`

What is actually in the modelling dataset, before any modelling happens.

The notebook is read top to bottom and answers, in order:

1. **Shape & schema** — how big, what columns, what types
2. **Integrity** — is the key unique, are there impossible values
3. **Missingness** — how much, where, and *whether it is random* (it is not)
4. **Target** — distribution of `market_value`, and why the model logs it
5. **Panel structure** — how many players, how many seasons each, lag coverage
6. **Univariate** — distributions of the inputs
7. **Relationships with the target** — age curve, minutes, xG, correlations, FBref
8. **Position-aware views** — the project benchmarks within position, so EDA does too
9. **Split sanity** — does train look like test
10. **Findings** — what carries into the model

Nothing here writes to disk.

## 0. Setup

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore")


def find_project_root(marker: str = "CLAUDE.md") -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"could not find {marker} at or above {here}")


BASE_PATH = Path(os.environ.get("TRANSFER_EDGE_ROOT") or find_project_root())
sys.path.insert(0, str(BASE_PATH))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

print(f"project root: {BASE_PATH}")

In [ ]:
# ── Chart style ──────────────────────────────────────────────────────────────
# A validated categorical order (fixed — slots are assigned by entity, never
# cycled) plus recessive chrome. Three of these slots sit below 3:1 against the
# surface, so every figure that uses them also prints its underlying table.

SURFACE = "#fcfcfb"
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # blue, orange, aqua, yellow, magenta
BLUE = SERIES[0]

# Correlations are polar (-1..+1): two opposing hues, neutral gray midpoint.
DIVERGING = LinearSegmentedColormap.from_list("blue_red", [
    "#0d366b", "#2a78d6", "#9ec5f4", "#f0efec", "#f3a9a8", "#e34948", "#8f2020",
])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "semibold", "axes.titlelocation": "left",
    "axes.titlecolor": INK, "axes.titlepad": 10,
    "axes.labelcolor": INK_2, "axes.labelsize": 9,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "grid.linestyle": "-",
    "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.direction": "out", "ytick.direction": "out",
    "legend.frameon": False, "legend.fontsize": 9, "legend.labelcolor": INK_2,
    "lines.linewidth": 2.0, "lines.markersize": 6,
    "figure.dpi": 110,
})

POS_ORDER = ["GK", "DEF", "MID", "ATT"]      # DM is folded into DEF (engineer.py)
POS_COLOR = dict(zip(POS_ORDER, SERIES))   # entity -> hue, fixed


def eur(v, _=None):
    """Axis formatter: 2_500_000 -> '€2.5M'."""
    if not np.isfinite(v):
        return ""
    for cut, suf in ((1e9, "B"), (1e6, "M"), (1e3, "k")):
        if abs(v) >= cut:
            return f"€{v / cut:,.1f}{suf}"
    return f"€{v:,.0f}"


def finish(ax, title=None, xlabel=None, ylabel=None, grid_axis="y"):
    """Apply the recessive-chrome conventions to one axes."""
    if title:
        ax.set_title(title)
    ax.set_xlabel(xlabel or "")
    ax.set_ylabel(ylabel or "")
    ax.grid(axis=grid_axis)
    ax.grid(axis="x" if grid_axis == "y" else "y", visible=False)
    return ax


print(f"{len(SERIES)} categorical slots, fixed order")

In [ ]:
from src.data.loader import load_panel
from src.features.engineer import (
    EXCLUDED_POSITIONS, FEATURE_COLS, PEAK_AGES, TRAIN_END, VAL_YEAR, build_features,
    fit_peak_ages, select_features,
)

panel = load_panel()
print(f"panel_final: {panel.shape[0]:,} rows x {panel.shape[1]} columns")

# Engineered view — build_features drops rows with no usable market value.
feat = build_features(panel)
print(f"after build_features: {feat.shape[0]:,} rows ({len(panel) - len(feat):,} dropped: no market value)")

HAS_IMPUTED = "market_value_imputed" in panel.columns
HAS_US_ID = "understat_id" in panel.columns
HAS_FBREF = "fb_tackles_won" in panel.columns

# Peak ages estimated on training seasons only (fit_peak_ages). Context only:
# no model feature uses them any more — age is the only age feature.
PEAK_FIT = fit_peak_ages(feat)

# The rows the model can learn from: no goalkeepers, no interpolated targets.
imputed = feat["market_value_imputed"].fillna(False).astype(bool) if HAS_IMPUTED else False
modelled = ~feat["pos_group"].isin(EXCLUDED_POSITIONS) & ~imputed
print(f"FBref joined: {HAS_FBREF}   rows usable by the model: {int(modelled.sum()):,} of {len(feat):,}")
print(f"imputation flags present: {HAS_IMPUTED}   understat_id present: {HAS_US_ID}")

## 1. Shape & schema

First question of any EDA: what is one row, and what is in it. Here a row should
be one **player-season** — that is the grain notebook 02 enforces.

In [ ]:
schema = pd.DataFrame({
    "dtype": panel.dtypes.astype(str),
    "non_null": panel.notna().sum(),
    "null_pct": (panel.isna().mean() * 100).round(1),
    "n_unique": panel.nunique(),
})
schema["example"] = [panel[c].dropna().iloc[0] if panel[c].notna().any() else None
                     for c in panel.columns]
schema

In [ ]:
print(f"rows            : {len(panel):,}")
print(f"columns         : {panel.shape[1]}")
print(f"memory          : {panel.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"seasons         : {panel.year.min()}–{panel.year.max()} ({panel.year.nunique()} of them)")
print(f"distinct players: {panel.player_id.nunique():,}")
print(f"distinct clubs  : {panel.club.nunique()}")

## 2. Integrity

Cheap assertions that catch a broken upstream run. These are the checks that
would have caught the duplicated-stats bug in the old merge.

In [ ]:
checks = []


def check(name, ok, detail=""):
    checks.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})


dup_key = panel.duplicated(subset=["year", "player_id"]).sum()
check("one row per (year, player_id)", dup_key == 0, f"{dup_key} duplicates")

check("no fully duplicated rows", panel.duplicated().sum() == 0)

const_cols = [c for c in panel.columns if panel[c].nunique(dropna=False) <= 1]
check("no constant columns", not const_cols, ", ".join(const_cols))

neg = {c: int((panel[c] < 0).sum()) for c in
       ["minutes", "games", "goals", "assists", "shots", "market_value"]
       if c in panel.columns}
check("no negative counts", all(v == 0 for v in neg.values()), str({k: v for k, v in neg.items() if v}))

bad_age = panel.age.dropna()
check("age within 14–45", bad_age.between(14, 45).all(),
      f"range {bad_age.min():.0f}–{bad_age.max():.0f}")

bad_h = panel.height.dropna()
check("height within 1.5–2.2 m", bad_h.between(1.5, 2.2).all(),
      f"range {bad_h.min():.2f}–{bad_h.max():.2f}")

mins = panel.minutes.dropna()
check("minutes <= 38 games x 90 + stoppage", (mins <= 3600).all(), f"max {mins.max():.0f}")

zero_mv = int((panel.market_value == 0).sum())
check("no zero market values", zero_mv == 0,
      f"{zero_mv} rows at exactly 0 — log(0) is -inf, so build_features drops them")

if "match_score" in panel:
    ms = panel.match_score.dropna()
    check("no match below threshold 85", (ms >= 85).all(), f"min {ms.min():.1f}")

if HAS_US_ID:
    reuse = (panel[panel.match_score.notna()]
             .groupby(["year", "understat_id"]).size().pipe(lambda s: (s > 1).sum()))
    check("each Understat player used once per season", reuse == 0, f"{reuse} reused")

pd.DataFrame(checks)

## 3. Missingness

Two separate questions, and the second one matters more.

**How much is missing** is a tidiness question. **Whether it is missing at
random** is a modelling question — if absence carries signal, imputing it away
destroys information.

In [ ]:
miss = (panel.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]

fig, ax = plt.subplots(figsize=(7.5, max(2.5, 0.28 * len(miss))))
y = np.arange(len(miss))
ax.barh(y, miss.values, color=BLUE, height=0.62)
ax.set_yticks(y, miss.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlim(0, max(100, miss.max() * 1.15))
for yi, v in zip(y, miss.values):          # direct labels — never a bare bar
    ax.text(v + 1.2, yi, f"{v:.1f}%", va="center", color=INK_2, fontsize=8.5)
finish(ax, "Missing values by column", xlabel="% of rows missing", grid_axis="x")
plt.tight_layout(); plt.show()

miss.round(1).to_frame("null_pct")

In [ ]:
# Columns arriving from Understat go missing together — they are one block,
# present only when the fuzzy join found a match.
us_cols = [c for c in ["games", "minutes", "goals", "xG", "assists", "xA", "shots",
                       "key_passes", "npg", "npxG", "xGChain", "xGBuildup"]
           if c in panel.columns]

pattern = (panel[us_cols].isna().sum(axis=1)
           .value_counts().sort_index()
           .rename_axis("n_missing_understat_cols").to_frame("rows"))
pattern["pct"] = (100 * pattern.rows / len(panel)).round(1)
print("Understat columns are all-or-nothing per row:")
pattern

### Is the missingness random?

Compare the target between rows that matched Understat and rows that did not.
If those two medians differ, missingness is **informative** — it encodes
something real about the player rather than a random gap.

In [ ]:
panel["has_stats"] = panel.match_score.notna()
by_stats = panel.groupby("has_stats").market_value.agg(
    rows="size", median="median", p25=lambda s: s.quantile(.25), p75=lambda s: s.quantile(.75)
)
print(by_stats.to_string())

ratio = by_stats.loc[True, "median"] / by_stats.loc[False, "median"]
print(f"\nplayers with stats are worth {ratio:.1f}x the median of players without")
print("\n-> NOT missing at random. Understat only lists players who appeared, so an")
print("   absent row means fringe / reserve / loaned out. XGBoost learns a direction")
print("   for NaN natively and will use this; median-imputing (as the Ridge baseline")
print("   does) erases it.")

## 4. The target

`market_value` in EUR — Transfermarkt's snapshot at the **end** of each season
(June of year+1; `value_date` says which), so the full-season stats come
before it. The model fits `log(market_value)`; this section is where
that choice gets justified rather than assumed.

In [ ]:
mv = panel.market_value.dropna()
mv_pos = mv[mv > 0]          # log() needs strictly positive; see the zero check above

desc = mv.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])
print(desc.apply(lambda v: f"{v:,.0f}").to_string())
print(f"\nnon-null      : {len(mv):,}   of which > 0: {len(mv_pos):,}")
print(f"skew (raw)    : {mv.skew():.2f}")
print(f"skew (logged) : {np.log(mv_pos).skew():.2f}")
print(f"max / median  : {mv.max() / mv.median():.0f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].hist(mv / 1e6, bins=60, color=BLUE)
axes[0].xaxis.set_major_formatter(lambda v, _: f"€{v:,.0f}M")
finish(axes[0], "Market value — raw EUR", ylabel="players")
axes[0].text(0.97, 0.9, f"skew {mv.skew():.1f}", transform=axes[0].transAxes,
             ha="right", color=INK_2, fontsize=9)

axes[1].hist(np.log(mv_pos), bins=60, color=BLUE)
finish(axes[1], "Market value — log scale", ylabel="players", xlabel="log(EUR)")
axes[1].text(0.97, 0.9, f"skew {np.log(mv_pos).skew():.1f}", transform=axes[1].transAxes,
             ha="right", color=INK_2, fontsize=9)

plt.tight_layout(); plt.show()
print("Raw values span three orders of magnitude with a long right tail; the log is")
print("near-symmetric. That is the whole argument for log(market_value) as the target,")
print("and for reading errors as percentages rather than euros.")

In [ ]:
# Value level by season — is there market-wide inflation the model should know about?
by_year = panel.groupby("year").market_value.agg(
    median="median", p25=lambda s: s.quantile(.25), p75=lambda s: s.quantile(.75), rows="size"
)

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.fill_between(by_year.index, by_year.p25, by_year.p75, color=BLUE, alpha=0.15, linewidth=0)
ax.plot(by_year.index, by_year["median"], color=BLUE, marker="o")
ax.yaxis.set_major_formatter(eur)
ax.set_xticks(by_year.index)
finish(ax, "Market value by season  ·  median, with interquartile band", xlabel="season")
plt.tight_layout(); plt.show()

by_year.assign(**{c: by_year[c].map("{:,.0f}".format) for c in ["median", "p25", "p75"]})

## 5. Panel structure

This is a panel, not a cross-section: the same player recurs across seasons.
How *balanced* that panel is determines how often lag features can exist at all.

In [ ]:
seasons_per = panel.groupby("player_id").size()
counts = seasons_per.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.bar(counts.index, counts.values, color=BLUE, width=0.62)
for x, v in counts.items():
    ax.text(x, v + counts.max() * 0.02, f"{v:,}", ha="center", color=INK_2, fontsize=8.5)
ax.set_xticks(counts.index)
ax.set_ylim(0, counts.max() * 1.12)
finish(ax, "Seasons observed per player", xlabel="seasons in the panel", ylabel="players")
plt.tight_layout(); plt.show()

one_season = 100 * (seasons_per == 1).mean()
print(f"{seasons_per.size:,} distinct players; {one_season:.0f}% appear in exactly one season.")
print("Every one of those rows has no previous season, so every lag feature is NaN there.")

In [ ]:
# Lag coverage by split — the features CLAUDE.md calls the strongest predictor.
feat["split"] = np.where(feat.year <= TRAIN_END, "train",
                         np.where(feat.year == VAL_YEAR, "val", "test"))
lag_cols = [c for c in ["log_value_lag1", "xG_p90_lag1", "value_growth"] if c in feat.columns]

lag_cov = (feat.groupby("split")[lag_cols].apply(lambda g: g.notna().mean() * 100)
           .reindex(["train", "val", "test"]))

x = np.arange(len(lag_cov))
w = 0.26
fig, ax = plt.subplots(figsize=(7.5, 3.4))
for i, col in enumerate(lag_cols):
    ax.bar(x + (i - 1) * w, lag_cov[col], width=w - 0.02, color=SERIES[i], label=col)
    for xi, v in zip(x + (i - 1) * w, lag_cov[col]):
        ax.text(xi, v + 1.5, f"{v:.0f}%", ha="center", color=INK_2, fontsize=8)
ax.set_xticks(x, lag_cov.index)
ax.set_ylim(0, 100)
ax.legend(loc="upper left", ncols=len(lag_cols))
finish(ax, "Lag feature coverage by split", ylabel="% of rows with a value")
plt.tight_layout(); plt.show()

lag_cov.round(1)

## 6. Univariate distributions

The inputs on their own terms, before any relationship to the target.

In [ ]:
num_cols = [c for c in ["age", "height", "minutes", "games", "goals", "assists",
                        "xG", "xA", "shots", "key_passes", "npxG", "xGChain", "xGBuildup"]
            if c in panel.columns]

panel[num_cols].describe().T.assign(
    null_pct=(panel[num_cols].isna().mean() * 100).round(1),
    skew=panel[num_cols].skew().round(2),
)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(13, 7.5))
for ax, col in zip(axes.flat, num_cols[:12]):
    s = panel[col].dropna()
    ax.hist(s, bins=40, color=BLUE)
    finish(ax, col, ylabel="")
    ax.tick_params(labelsize=8)
for ax in axes.flat[len(num_cols[:12]):]:
    ax.set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# Categoricals — how concentrated are they?
for col in ["position", "nationality", "foot", "club"]:
    if col not in panel.columns:
        continue
    vc = panel[col].value_counts(dropna=False)
    top = vc.head(8)
    print(f"\n{col}  ({vc.size} distinct, top 8 = {100 * top.sum() / len(panel):.0f}% of rows)")
    print((top.to_frame("rows").assign(pct=(100 * top / len(panel)).round(1))).to_string())

In [ ]:
# Position groups — the unit the whole product benchmarks against.
pos = feat.pos_group.value_counts().reindex(POS_ORDER)
split_pos = pd.crosstab(feat.pos_group, feat.split).reindex(
    index=POS_ORDER, columns=["train", "val", "test"])

fig, ax = plt.subplots(figsize=(7.5, 3.4))
bottom = np.zeros(len(POS_ORDER))
for i, sp in enumerate(["train", "val", "test"]):
    vals = split_pos[sp].values
    ax.bar(POS_ORDER, vals, bottom=bottom, width=0.6, label=sp,
           color=SERIES[i], edgecolor=SURFACE, linewidth=2)   # 2px surface gap
    bottom += vals
for xi, total in enumerate(bottom):
    ax.text(xi, total + bottom.max() * 0.02, f"{int(total):,}", ha="center",
            color=INK_2, fontsize=8.5)
ax.set_ylim(0, bottom.max() * 1.12)
ax.legend(loc="upper right", ncols=3)
finish(ax, "Rows per position group, by split", ylabel="player-seasons")
plt.tight_layout(); plt.show()

split_pos.assign(total=split_pos.sum(axis=1))

## 7. Relationships with the target

### Age

Where does value peak, per position? The grey line is the peak `fit_peak_ages()`
estimates on the **training seasons only**; the dot is where median value peaks
across all seasons. The model itself uses only `age` — trees draw this curve on
their own — so this is context, not a feature.
Each panel is its own small multiple rather than four colors on one axes.

In [ ]:
feat["log_value"] = np.log(feat.market_value)

fig, axes = plt.subplots(2, 2, figsize=(11, 6.4), sharey=True)
for ax, grp in zip(axes.flat, POS_ORDER):
    sub = feat[feat.pos_group == grp]
    by_age = sub.groupby(sub.age.round())["log_value"].agg(["median", "size"])
    by_age = by_age[by_age["size"] >= 10]          # drop ages with too little support

    ax.plot(by_age.index, by_age["median"], color=POS_COLOR[grp], marker="o", markersize=4)
    fitted = PEAK_FIT.get(grp)
    ax.axvline(fitted, color=MUTED, linewidth=1)
    ax.text(fitted + 0.25, ax.get_ylim()[0], f" fitted on train: {fitted}",
            color=MUTED, fontsize=8, va="bottom")
    best = by_age["median"].idxmax()
    ax.scatter([best], [by_age["median"].max()], s=60, color=POS_COLOR[grp],
               zorder=5, edgecolor=SURFACE, linewidth=2)
    note = "  · not modelled" if grp in EXCLUDED_POSITIONS else ""
    finish(ax, f"{grp}  (observed peak {best:.0f}, fitted {fitted}){note}",
           xlabel="age", ylabel="median log(value)")
    ax.tick_params(labelsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# The same thing as a table. "prior" is the conventional wisdom (PEAK_AGES),
# "fitted_train" is what the features use, "observed_all" peeks at every season
# — shown for context only, it must never feed back into the model.
peak_rows = []
for grp in POS_ORDER:
    sub = feat[feat.pos_group == grp]
    by_age = sub.groupby(sub.age.round())["log_value"].agg(["median", "size"])
    by_age = by_age[by_age["size"] >= 10]
    observed = int(by_age["median"].idxmax())
    peak_rows.append({
        "pos_group": grp,
        "prior": PEAK_AGES.get(grp),
        "fitted_train": PEAK_FIT.get(grp),
        "observed_all": observed,
        "gap": observed - PEAK_FIT.get(grp),
        "rows": len(sub),
    })
pd.DataFrame(peak_rows).set_index("pos_group")

### Minutes, performance and value

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

for ax, (col, label) in zip(axes, [("minutes", "minutes played"),
                                   ("xG_p90", "xG per 90"),
                                   ("xGBuildup_p90", "xGBuildup per 90")]):
    sub = feat[[col, "log_value"]].dropna()
    ax.scatter(sub[col], sub.log_value, s=5, alpha=0.18, color=BLUE, linewidths=0)
    # binned median on top, so the trend is readable through the cloud
    bins = pd.qcut(sub[col], 12, duplicates="drop")
    trend = sub.groupby(bins, observed=True).agg(x=(col, "median"), y=("log_value", "median"))
    ax.plot(trend.x, trend.y, color="#eb6834", marker="o", markersize=4)
    finish(ax, f"log(value) vs {label}", xlabel=label, ylabel="log(value)")
    ax.tick_params(labelsize=8)

plt.tight_layout(); plt.show()
print("Orange line: median log(value) within each of 12 equal-count bins of the x variable.")

In [ ]:
# Correlation with the target, across the engineered feature set.
X = select_features(feat)
corr_target = X.corrwith(feat.log_value).dropna().sort_values()
top = pd.concat([corr_target.head(10), corr_target.tail(12)])

fig, ax = plt.subplots(figsize=(7.5, 6.2))
colors = [DIVERGING(0.5 + 0.5 * np.clip(v, -1, 1)) for v in top.values]
y = np.arange(len(top))
ax.barh(y, top.values, color=colors, height=0.68)
ax.set_yticks(y, top.index, fontsize=8.5)
ax.axvline(0, color=AXIS, linewidth=0.8)
for yi, v in zip(y, top.values):
    ax.text(v + (0.015 if v >= 0 else -0.015), yi, f"{v:+.2f}",
            va="center", ha="left" if v >= 0 else "right", color=INK_2, fontsize=8)
ax.set_xlim(top.min() - 0.12, top.max() + 0.12)
finish(ax, "Correlation with log(market_value)", xlabel="Pearson r", grid_axis="x")
plt.tight_layout(); plt.show()

corr_target.round(3).to_frame("r_with_log_value")

In [ ]:
# Feature-to-feature correlation — where is the redundancy?
core = [c for c in FEATURE_COLS if not c.startswith(("posgrp_", "natgrp_"))]
cm = X[core].corr()

fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(cm, cmap=DIVERGING, vmin=-1, vmax=1)
ax.set_xticks(range(len(core)), core, rotation=90, fontsize=7.5)
ax.set_yticks(range(len(core)), core, fontsize=7.5)
ax.grid(False)
cb = fig.colorbar(im, ax=ax, shrink=0.75)
cb.set_label("Pearson r", color=INK_2, fontsize=9)
cb.outline.set_edgecolor(AXIS)
ax.set_title("Feature correlation matrix")
plt.tight_layout(); plt.show()

pairs = (cm.where(np.triu(np.ones(cm.shape), 1).astype(bool))
         .stack().sort_values(key=abs, ascending=False))
print("Most redundant pairs (|r| > 0.9):")
pairs[abs(pairs) > 0.9].round(3).to_frame("r")

### FBref: defensive actions and fouls

Notebook 05 adds the only defensive stats FBref still publishes — tackles won
and interceptions — plus fouls committed, fouls drawn and crosses, all per 90.
The question is whether they separate cheap from expensive players *within* a
position. Each row of panels is one stat; each column is one position; the line
is median log(value) across quintiles of the stat.

In [ ]:
if not HAS_FBREF:
    print("fbref_matched.parquet not found — run notebook 05 first.")
else:
    fb_cols = ["tackles_won_p90", "interceptions_p90", "fouls_p90", "fouled_p90", "crosses_p90"]
    cov = (feat.groupby("pos_group")[fb_cols[0]].apply(lambda s: s.notna().mean() * 100)
           .reindex(POS_ORDER).round(0))
    print("FBref coverage by position (% of rows with a match):")
    print(cov.to_string())

    stats = [("def_actions_p90", "tackles + interceptions / 90"), ("fouled_p90", "fouls drawn / 90")]
    groups = [g for g in POS_ORDER if g not in EXCLUDED_POSITIONS]
    fig, axes = plt.subplots(len(stats), len(groups), figsize=(12, 6), sharey=True)
    rows = []
    for r, (col, label) in enumerate(stats):
        for c, grp in enumerate(groups):
            ax = axes[r, c]
            sub = feat[(feat.pos_group == grp) & (feat.minutes >= 450)][[col, "log_value"]].dropna()
            q = pd.qcut(sub[col], 5, labels=False, duplicates="drop")
            line = sub.groupby(q)["log_value"].median()
            ax.plot(line.index + 1, line.values, color=POS_COLOR[grp], marker="o", markersize=5)
            ax.set_xticks(range(1, 6))
            finish(ax, f"{grp} — {label}", xlabel="quintile (1 = lowest)",
                   ylabel="median log(value)" if c == 0 else None)
            ax.tick_params(labelsize=8)
            rows.append({"stat": col, "pos_group": grp, "n": len(sub),
                         "Q1": round(line.iloc[0], 2), "Q5": round(line.iloc[-1], 2),
                         "spread": round(line.iloc[-1] - line.iloc[0], 2)})
    plt.tight_layout(); plt.show()
    print("Players with >= 450 minutes. spread = Q5 - Q1 in log(value); +0.1 is roughly +10% value.")
    display(pd.DataFrame(rows).set_index(["stat", "pos_group"]))

**What the model made of it** (see CLAUDE.md): adding the six FBref features
moved test RMSE(log) 0.730 → 0.714 and MAPE 32.1% → 31.8%, but both 95%
bootstrap intervals include zero. The model leaned on fouls and fouls drawn,
not on tackles and interceptions — defenders' error did not move.

## 8. Position-aware views

The product benchmarks players against positional peers, never the whole squad.
The same discipline applies when looking at the data.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
data = [feat.loc[feat.pos_group == g, "log_value"].dropna() for g in POS_ORDER]
bp = ax.boxplot(data, tick_labels=POS_ORDER, patch_artist=True, widths=0.55,
                medianprops=dict(color=INK, linewidth=1.6),
                whiskerprops=dict(color=AXIS), capprops=dict(color=AXIS),
                flierprops=dict(marker="o", markersize=3, markerfacecolor=MUTED,
                                markeredgecolor="none", alpha=0.35))
for patch, g in zip(bp["boxes"], POS_ORDER):
    patch.set_facecolor(POS_COLOR[g]); patch.set_alpha(0.75); patch.set_edgecolor(SURFACE)
    patch.set_linewidth(2)
finish(ax, "log(market_value) by position group", ylabel="log(EUR)")
plt.tight_layout(); plt.show()

feat.groupby("pos_group").market_value.agg(
    rows="size", median="median", p25=lambda s: s.quantile(.25), p75=lambda s: s.quantile(.75)
).reindex(POS_ORDER).assign(
    median=lambda d: d["median"].map("{:,.0f}".format),
    p25=lambda d: d.p25.map("{:,.0f}".format),
    p75=lambda d: d.p75.map("{:,.0f}".format),
)

In [ ]:
# Why percentile features are computed within (year, position): the same raw
# number means different things per group.
comp = feat.groupby("pos_group")[["xG_p90", "xA_p90", "minutes", "goals_p90"]].median()
print("Median of each stat, by position group — note how far apart the scales are:")
comp.reindex(POS_ORDER).round(3)

### Positional versatility

`position_us` is a multi-label code — `"D F M S"` means the player turned out in
defence, attack and midfield, and also came off the bench. Counting the distinct
positions (excluding `S`, which is a substitute appearance rather than a
position) gives `position_versatility`.

The question EDA has to answer is whether it is anything more than a proxy for
playing time: a player who is never on the pitch cannot be seen in two roles.

In [ ]:
vers = feat[["position_versatility", "minutes", "market_value", "pos_group"]].dropna(
    subset=["position_versatility"])

dist = vers.groupby("position_versatility").agg(
    rows=("market_value", "size"),
    median_minutes=("minutes", "median"),
    median_value=("market_value", "median"),
)
dist["pct_of_matched"] = (100 * dist.rows / len(vers)).round(1)

print("Distribution (only rows that matched Understat):")
print(dist.assign(median_value=lambda d: d.median_value.map("{:,.0f}".format)).to_string())
print()

sub_only = vers[vers.position_versatility == 0]
print(f"0 positions = appeared only as a substitute: {len(sub_only)} rows, median "
      f"{sub_only.minutes.median():.0f} minutes.")
print("That is a real observation, not missing data — the player featured but never")
print("in a recorded position.")
print()
print("Goalkeepers are always exactly 1, so the feature is constant for them:")
print(vers[vers.pos_group == "GK"].position_versatility.value_counts().to_string())

In [ ]:
# The test that matters: does the premium survive inside a minutes band?
band = vers[vers.position_versatility.between(1, 3)].copy()
band["minutes_q"] = pd.qcut(band.minutes, 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
piv = band.pivot_table(index="minutes_q", columns="position_versatility",
                       values="market_value", aggfunc="median", observed=True)

x = np.arange(len(piv))
w = 0.26
fig, ax = plt.subplots(figsize=(8, 3.8))
for i, lv in enumerate(piv.columns):
    ax.bar(x + (i - 1) * w, piv[lv], width=w - 0.02, color=SERIES[i],
           label=f"{int(lv)} position" + ("s" if lv > 1 else ""))
    for xi, v in zip(x + (i - 1) * w, piv[lv]):
        if np.isfinite(v):
            ax.text(xi, v * 1.03, eur(v), ha="center", color=INK_2, fontsize=8)
ax.set_xticks(x, piv.index)
ax.yaxis.set_major_formatter(eur)
ax.set_ylim(0, np.nanmax(piv.values) * 1.18)
ax.legend(loc="upper left", ncols=3)
finish(ax, "Median market value by playing time and versatility",
       xlabel="minutes played (quartile)")
plt.tight_layout(); plt.show()

piv.map(lambda v: f"{v:,.0f}" if pd.notna(v) else "-")

In [ ]:
r = np.corrcoef(band.position_versatility, np.log(band.market_value))[0, 1]
lift = (piv[2.0] / piv[1.0] - 1) * 100

print(f"correlation with log(value), rows with at least one position: r = {r:+.3f}")
print()
print("premium of 2 positions over 1, within each minutes quartile:")
print(lift.map(lambda v: f"{v:+.0f}%").to_string())
print()
print("-> The premium holds in every band, so this is not just a playing-time proxy.")
print("   3 positions is noisier (84 rows) and only separates from 2 at high minutes.")

## 9. Split sanity

The split is temporal, so train and test are different *eras*, not a random
sample. Anything that drifts between them is a thing the model will extrapolate
on rather than interpolate.

In [ ]:
drift = feat.groupby("split")[["age", "minutes", "xG_p90", "market_value",
                               "club_med_value"]].median().reindex(["train", "val", "test"])
drift["rows"] = feat.split.value_counts().reindex(["train", "val", "test"])
print("Median by split:")
print(drift.to_string())

print("\nSeasons per split:")
print(feat.groupby("split").year.agg(["min", "max", "size"]).reindex(["train", "val", "test"]).to_string())

if HAS_IMPUTED:
    print("\nRows with an INTERPOLATED target — never used by the model (train, val or test):")
    print(feat.groupby("split").market_value_imputed.sum().reindex(["train", "val", "test"]).to_string())

In [ ]:
# Target level by split — a temporal split means the model predicts a future
# price level from a past one.
fig, ax = plt.subplots(figsize=(7.5, 3.4))
for i, sp in enumerate(["train", "val", "test"]):
    s = np.log(feat.loc[feat.split == sp, "market_value"].dropna())
    ax.hist(s, bins=45, histtype="step", linewidth=2, color=SERIES[i], label=f"{sp} (n={len(s):,})")
ax.legend(loc="upper left")
finish(ax, "log(market_value) distribution by split", xlabel="log(EUR)", ylabel="players")
plt.tight_layout(); plt.show()

## 10. Findings

Auto-generated from what the notebook just computed, so it cannot drift away
from the data.

In [ ]:
n_players = panel.player_id.nunique()
one_season_pct = 100 * (panel.groupby("player_id").size() == 1).mean()
stats_pct = 100 * panel.match_score.notna().mean()
ratio = (panel.groupby("has_stats").market_value.median().pipe(lambda s: s[True] / s[False]))
lag_pct = 100 * feat.log_value_lag1.notna().mean()
full = int((X.notna().mean() == 1).sum())
peak_gaps = {r["pos_group"]: r["gap"] for r in peak_rows}

print(f"""
GRAIN       {len(panel):,} player-seasons, {panel.year.min()}–{panel.year.max()}, {n_players:,} distinct players.
            {one_season_pct:.0f}% of players appear in only one season.

TARGET      Spans {panel.market_value.max() / panel.market_value.median():.0f}x from median to max; raw skew
            {panel.market_value.skew():.1f} vs {np.log(mv_pos).skew():.1f} logged. Log target is the right call.

COVERAGE    {stats_pct:.0f}% of rows carry Understat stats. Only {full} of {X.shape[1]} model features are
            fully populated; lag value present in {lag_pct:.0f}% of rows.

MISSINGNESS NOT random — matched players are worth {ratio:.1f}x the median of unmatched ones.
            Absence is signal. Keep NaN for tree models; do not median-impute.

POSITION    Smallest group is {feat.pos_group.value_counts().idxmin()} with only
            {feat.pos_group.value_counts().min():,} rows — thin for a position-specific model.

VERSATILITY Present for {100 * feat.position_versatility.notna().mean():.0f}% of rows. Median value by distinct positions:
            {dict(feat.groupby('position_versatility').market_value.median().astype(int))}
            Premium holds within minutes bands, so it is not a playing-time proxy.

PEAK AGE    Observed (all seasons) minus fitted-on-train peak, by group: {peak_gaps}
            (context only — the model uses age directly)
""")

### What to carry into modelling

- **Keep NaN as NaN for the tree model.** Missingness is informative here, and
  median imputation throws that away. Any comparison against a median-imputing
  baseline is partly measuring the imputation, not the model.
- **Goalkeepers are not modelled** and **DM is folded into DEF** — the smallest
  groups could not support a model of their own.
- **`age` is the only age feature.** age², years-to-peak and prime-window were
  the same information for a tree and only split the age effect across SHAP bars;
  removing them cost no measurable accuracy.
- **Interpolated targets never reach the model**, not as target, lag or club
  median: they are built from next season's value.
- **The split is temporal**, so drift in section 9 is extrapolation, not noise.